# 예제 V · Token Merging (ToMe) (수강생용)

같은 허스키 이미지에서 ToMe의 병합 설정 `r`만 변경합니다. ToMe를 적용하지 않은 것과 같은 조건인 `r=0`을 기준으로, 마지막 블록의 Token 수와 Top-5 클래스 목록을 비교합니다.

## 1. 라이브러리, 모델, 예제 이미지 준비

셀 1–3을 처음에 한 번 실행합니다. 정답용 예제에 쓰인 ViT-Base 가중치는 첫 실행에서 내려받습니다.

**코드 셀 1 · 실행 장치와 timm 준비**

CPU 또는 GPU를 선택하고 ToMe와 호환되는 timm 0.9.10이 필요하면 설치합니다.

In [ ]:
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys
import time

import torch
import matplotlib.pyplot as plt
from PIL import Image

torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ASSET_ROOT = Path(os.environ.get("AX_LAB_CACHE", "./student_lab_cache"))
ASSET_ROOT.mkdir(parents=True, exist_ok=True)
try:
    timm_version = importlib.metadata.version("timm")
except importlib.metadata.PackageNotFoundError:
    timm_version = None
# 공식 ToMe 구현과 호환되는 timm 버전을 준비한다.
if timm_version != "0.9.10":
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "timm==0.9.10"])
print("실행 장치:", DEVICE)

**코드 셀 2 · ToMe 및 데이터 다운로드 함수 정의**

파일 캐시를 재사용하고 지정한 GitHub 커밋의 ZIP을 경로 검사 후 푸는 함수를 정의합니다.

In [ ]:
from urllib.request import urlretrieve
from zipfile import ZipFile
import sys

# 체크포인트와 소스를 캐시에 보관한다. 다운로드가 완료되면 임시 파일을 정상 파일 이름으로 바꾼다.
def fetch(url, filename):
    destination = ASSET_ROOT / filename
    if not destination.is_file():
        pending = destination.with_suffix(destination.suffix + ".download")
        try:
            urlretrieve(url, pending)
            pending.replace(destination)
        finally:
            pending.unlink(missing_ok=True)
    return destination

# 고정한 커밋의 ZIP을 받고, 각 항목의 경로를 검사한 뒤 추출한다.
def fetch_repository(owner, repository, revision, filename):
    archive = fetch(f"https://codeload.github.com/{owner}/{repository}/zip/{revision}", filename)
    root = ASSET_ROOT / f"{repository}-{revision}"
    if not root.exists():
        with ZipFile(archive) as source_zip:
            for entry in source_zip.infolist():
                destination = (ASSET_ROOT / entry.filename).resolve()
                if not destination.is_relative_to(root.resolve()):
                    raise ValueError(f"Unexpected archive path: {entry.filename}")
            source_zip.extractall(ASSET_ROOT)
    return root.resolve()

**코드 셀 3 · ViT-Base와 허스키 이미지 준비**

사전 학습 ViT-Base에 ToMe를 적용하고 허스키 이미지를 전처리하며 ImageNet 클래스 이름을 불러옵니다.

In [ ]:
import json
import timm
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode

# 고정한 ToMe 소스와 예제 Husky 이미지를 캐시에서 가져온다.
revision = "af95e4b1befa172dadccd8c81e223b10090f9579"
tome_root = fetch_repository("facebookresearch", "ToMe", revision, "tome.zip")
sys.path.insert(0, str(tome_root))
import tome

# 정답용과 같은 사전 학습 ViT-Base를 불러와 ToMe를 적용한다.
model = timm.create_model("vit_base_patch16_224", pretrained=True)
model.to(DEVICE).eval()
tome.patch.timm(model)

image = Image.open(tome_root / "examples" / "images" / "husky.png").convert("RGB")
# 정답용 예제와 동일한 크기 조정과 정규화를 적용한다.
input_size = model.default_cfg["input_size"][1]
resize_and_crop = transforms.Compose([
    transforms.Resize(int((256 / 224) * input_size), interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(input_size),
])
transform = transforms.Compose([
    resize_and_crop,
    transforms.ToTensor(),
    transforms.Normalize(model.default_cfg["mean"], model.default_cfg["std"]),
])
image_tensor = transform(image).unsqueeze(0).to(DEVICE)
# 같은 전처리의 정규화 전 이미지를 사용해 실제 모델 입력 위치에 경계를 표시한다.
model_crop = resize_and_crop(image)
measurement_cache = {}
labels_file = fetch(
    "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json",
    "imagenet_class_index.json"
)
class_names = json.loads(labels_file.read_text(encoding="utf-8"))

plt.figure(figsize=(3.2, 3.2))
plt.imshow(image)
plt.axis("off")
plt.show()

## 2. TODO · Token 병합 설정 바꾸기

- **수정할 값** · 바로 아래 **코드 셀 4**에서 `MERGE_TOKENS`의 `None` 자리에 **8**을 직접 입력하고 실행합니다. 이어서 **16**을 입력해 셀 4만 다시 실행합니다.
- **실행 후 확인** · 같은 허스키 이미지에서 기준 `r=0`과 비교해 마지막 블록의 Token 수, Top-5 클래스 이름과 순위가 어떻게 달라지는지 확인합니다.

**코드 셀 4 · Token 병합량과 Top-5 예측 비교**

MERGE_TOKENS를 바꿔 같은 허스키 이미지에서 기준 r=0 대비 마지막 블록의 Token 수와 Top-5 클래스 및 확률을 비교합니다.

In [ ]:
MERGE_TOKENS = None  # TODO
if type(MERGE_TOKENS) is not int or not 0 <= MERGE_TOKENS <= 16:
    raise ValueError("TODO: 코드 셀 4의 MERGE_TOKENS에 0부터 16 사이의 정수를 직접 입력하세요.")

# 기준과 선택값의 결과를 저장하고, 선택 실습에서도 같은 추론을 다시 하지 않는다.
@torch.inference_mode()
def get_tome_result(merge_tokens):
    if merge_tokens in measurement_cache:
        return measurement_cache[merge_tokens]

    block_tokens = []
    def count_tokens(_module, _args, output):
        block_tokens.append(output.shape[1])

    # 선택한 r에서 원본 Patch와 마지막 Block Token의 연결을 추적한다.
    trace = model._tome_info
    previous_r, previous_trace = model.r, trace["trace_source"]
    model.r, trace["trace_source"] = merge_tokens, True
    hooks = [block.register_forward_hook(count_tokens) for block in model.blocks]
    try:
        probabilities = model(image_tensor).softmax(1)
        source = trace["source"]
    finally:
        for hook in hooks:
            hook.remove()
        model.r, trace["trace_source"] = previous_r, previous_trace

    rows, cols = model.patch_embed.grid_size
    prefix_tokens = model.num_prefix_tokens
    # 병합이 없을 때 ToMe는 추적 행렬을 만들지 않으며, 각 Patch가 자체 Token을 유지한다.
    if source is None:
        patch_groups = torch.arange(rows * cols)
    else:
        image_sources = source[0, prefix_tokens:, prefix_tokens:]
        patch_groups = image_sources.argmax(dim=0).cpu()

    values, indices = probabilities[0].topk(5)
    result = {
        "top5": [(class_names[str(i)][1], p) for p, i in zip(values.tolist(), indices.tolist())],
        "input_tokens": rows * cols + prefix_tokens,
        "block_tokens": tuple(block_tokens),
        "patch_groups": patch_groups.reshape(rows, cols),
        "prefix_tokens": prefix_tokens,
    }
    measurement_cache[merge_tokens] = result
    return result


# 빠른 비교에서는 기준 r=0과 입력한 r만 실행한다.
conditions = list(dict.fromkeys((0, MERGE_TOKENS)))
measured = {r: get_tome_result(r) for r in conditions}
results = {r: (run["top5"], run["block_tokens"][-1]) for r, run in measured.items()}
for r, (top5, tokens) in results.items():
    print(f"r={r} | 마지막 블록 Token 수: {tokens} (클래스 Token 포함)")
    for rank, (name, probability) in enumerate(top5, 1):
        print(f"  {rank}. {name:<25} {probability:.2%}")

if len(results) == 2:
    before_names = [name for name, _ in results[0][0]]
    after_names = [name for name, _ in results[MERGE_TOKENS][0]]
    changed = [str(i) for i, (a, b) in enumerate(zip(before_names, after_names), 1) if a != b]
    print("기준과 다른 순위:", ", ".join(changed) if changed else "없음")

values = [results[r][1] for r in conditions]
fig, ax = plt.subplots(figsize=(5.5, 3.2), constrained_layout=True)
bars = ax.bar([f"r={r}" for r in conditions], values, color=["#64748b", "#167d9a"][:len(values)])
ax.bar_label(bars, padding=3)
ax.set_ylim(0, max(values) * 1.18)
ax.set_ylabel("Tokens after the last block")
plt.show()

## 3. 선택 실습 · 병합 영역과 Block별 결과 한눈에 보기

셀 1–4를 실행한 상태에서 **코드 셀 5**의 `SHOW_OVERVIEW`를 체크합니다. 같은 공식 예제 허스키 이미지의 **224×224 모델 입력 영역**에서 `r=0, 8, 16`을 비교합니다. 셀 4나 이전 실행에서 얻은 결과를 재사용하고 아직 측정하지 않은 조건만 추론합니다. 셀 3에서 이미지를 바꾸거나 다시 준비하면 결과도 초기화됩니다.

- **병합 영역** · 흰 선은 마지막 Block의 Token 경계입니다. 노란 테두리로 표시한 같은 위치의 원본 Patch가 속한 최종 Token의 모든 Patch를 붉게 표시합니다. 떨어진 Patch도 같은 Token으로 병합될 수 있습니다.
- **Block별 Token 수** · 0은 모델 입력이며, 모든 점은 분류에 쓰이는 `[CLS]` Token을 포함합니다. 병합 영역에 적힌 이미지 Patch Token 수는 `[CLS]`를 제외합니다.
- **Top-5** · 같은 입력 이미지에 대한 다섯 클래스의 순위와 예측 확률을 비교합니다. 이 확률은 전체 데이터셋의 정확도가 아닙니다.

In [ ]:
# @title 선택 실습 · 병합 영역, Block별 Token 수, Top-5 보기
SHOW_OVERVIEW = False  # @param {type:"boolean"}

from IPython.display import Markdown, display
from matplotlib.collections import LineCollection, PatchCollection
from matplotlib.patches import Rectangle


def draw_token_groups(axis, merge_tokens, run):
    # 동일한 원본 Patch 위치를 추적해 각 r에서 그 Patch가 속한 Token을 강조한다.
    groups = run["patch_groups"]
    rows, cols = groups.shape
    anchor_row, anchor_col = rows // 2, (cols - 1) // 2
    patch_width = model_crop.width / cols
    patch_height = model_crop.height / rows
    highlighted = (groups == groups[anchor_row, anchor_col]).nonzero().tolist()

    axis.imshow(model_crop, extent=(0, model_crop.width, model_crop.height, 0))
    shading = [Rectangle((col * patch_width, row * patch_height), patch_width, patch_height)
               for row, col in highlighted]
    axis.add_collection(PatchCollection(shading, facecolor="#D60000", edgecolor="none", alpha=.43))

    # 서로 다른 Token에 속한 인접 Patch 사이에만 경계선을 그린다.
    grid = groups.tolist()
    boundaries = []
    for row, group_row in enumerate(grid):
        for col, group in enumerate(group_row):
            x, y = col * patch_width, row * patch_height
            if col and group_row[col - 1] != group:
                boundaries.append(((x, y), (x, y + patch_height)))
            if row and grid[row - 1][col] != group:
                boundaries.append(((x, y), (x + patch_width, y)))
    axis.add_collection(LineCollection(boundaries, colors="white",
                                       linewidths=.45 if merge_tokens == 0 else .85))
    axis.add_patch(Rectangle((anchor_col * patch_width, anchor_row * patch_height),
                             patch_width, patch_height, fill=False, edgecolor="#FFF0D7", linewidth=1.15))
    axis.set(xlim=(0, model_crop.width), ylim=(model_crop.height, 0), title=f"r={merge_tokens}")
    axis.axis("off")
    image_tokens = run["block_tokens"][-1] - run["prefix_tokens"]
    return image_tokens, len(highlighted)


if SHOW_OVERVIEW:
    comparison = {r: get_tome_result(r) for r in (0, 8, 16)}
    display(Markdown("### 같은 224×224 입력에서 마지막 Block의 Token 병합 영역"))
    fig, axes = plt.subplots(1, len(comparison), figsize=(10, 3.8), constrained_layout=True)
    for axis, (r, run) in zip(axes, comparison.items()):
        image_tokens, patch_count = draw_token_groups(axis, r, run)
        axis.text(.5, -.04, f"{image_tokens} image tokens | {patch_count} highlighted patches",
                  transform=axis.transAxes, ha="center", va="top", fontsize=9)
    plt.show()

    display(Markdown("### Block별 Token 수 · 입력부터 마지막 Block까지"))
    fig, axis = plt.subplots(figsize=(8.5, 3.5), constrained_layout=True)
    for (r, run), color, marker in zip(comparison.items(), ("#666666", "#28738E", "#BA272D"), ("o", "s", "^")):
        counts = (run["input_tokens"], *run["block_tokens"])
        axis.plot(range(len(counts)), counts, label=f"r={r} · final {counts[-1]}",
                  color=color, marker=marker, markersize=4)
    axis.set(xticks=range(len(counts)), xlabel="Block (0 = model input)", ylabel="Tokens including [CLS]")
    axis.grid(axis="y", alpha=.25)
    axis.legend()
    plt.show()

    lines = ["### Top-5 클래스와 예측 확률", "", "| 순위 | r=0 | r=8 | r=16 |", "| ---: | --- | --- | --- |"]
    for rank, predictions in enumerate(zip(*(run["top5"] for run in comparison.values())), 1):
        values = [f"{name} · {probability:.2%}" for name, probability in predictions]
        lines.append(f"| {rank} | " + " | ".join(values) + " |")
    display(Markdown("\n".join(lines)))

## 추가로 바꿔볼 설정 · 선택 실습

- 같은 이미지에서 `MERGE_TOKENS`를 4 또는 12로 바꿉니다. 최종 Token 수와 Top-5 순서를 기준값 r=0과 비교합니다.
- 병합 설정을 고정하고 셀 3의 입력 이미지 파일만 다른 이미지로 바꿉니다. 새 이미지에서 r=0과 선택값을 비교합니다.